# Original versus width-ablated double-descent figures

Run the setup cells once, then run the cell for the figure you want to inspect. **Each numbered figure has its own cell, and each released source sweep has its own cell below.** Every panel displays the original full-width plot on the left and a freshly generated width-ablated plot on the right, using the same downloaded Parquet data and matched vertical or color scales. There is no figure selector.

The default ablation keeps widths **`2, 4, 8, 12, 16, 24, 32, 40, 48, 56, 64`**. Change `ABLATION_WIDTHS` in the helper cell to use another grid. Each figure cell shows 1%, 5%, 10%, and final saved measurement horizons; change `SELECTED_HORIZONS` if you only want some. These percentages refer to saved `measurement_index` positions, **not verified epochs or optimizer steps**. The main ResNet archives have 3,999 positions, so the early cutoffs contain 40, 200, and 400 positions.

This notebook reads the authors' released runs and does not train models. Translation results are endpoint-only and are outside this vision-history notebook. Run with the project `.venv` kernel.


In [ ]:
from io import BytesIO
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

project_dir = Path.cwd()
if (project_dir / 'Assignment 2 - Group' / 'their_plots' / 'short_horizon.py').is_file():
    plots_dir = project_dir / 'Assignment 2 - Group' / 'their_plots'
elif (project_dir / 'their_plots' / 'short_horizon.py').is_file():
    plots_dir = project_dir / 'their_plots'
elif (project_dir / 'short_horizon.py').is_file():
    plots_dir = project_dir
else:
    raise FileNotFoundError('Run this notebook from the project root, Assignment 2 - Group, or their_plots')

sys.path.insert(0, str(plots_dir))
from short_horizon import SPARSE_WIDTHS, parse_widths, render_figure

data_dir = plots_dir.parent / 'their-results' / 'hf_dataset'
print(f'Published vision data: {data_dir}')


In [ ]:
HORIZONS = (('1%', 0.01), ('5%', 0.05), ('10%', 0.10), ('Final', 1.0))


def align_scales(original: plt.Figure, ablated: plt.Figure) -> None:
    for original_axis, ablated_axis in zip(original.axes, ablated.axes, strict=True):
        if original_axis.lines and ablated_axis.lines:
            lower = min(original_axis.get_ylim()[0], ablated_axis.get_ylim()[0])
            upper = max(original_axis.get_ylim()[1], ablated_axis.get_ylim()[1])
            original_axis.set_ylim(lower, upper)
            ablated_axis.set_ylim(lower, upper)
        if original_axis.images and ablated_axis.images:
            original_image = original_axis.images[0]
            ablated_image = ablated_axis.images[0]
            lower = min(original_image.get_clim()[0], ablated_image.get_clim()[0])
            upper = max(original_image.get_clim()[1], ablated_image.get_clim()[1])
            original_image.set_clim(lower, upper)
            ablated_image.set_clim(lower, upper)


def plot_image(figure: plt.Figure) -> widgets.Image:
    image_buffer = BytesIO()
    figure.savefig(image_buffer, format='png', dpi=110, bbox_inches='tight')
    plt.close(figure)
    return widgets.Image(value=image_buffer.getvalue(), format='png', width=610)


def make_comparison(
    figure_name: str,
    ablated_widths: tuple[int, ...],
    horizons: tuple[float, ...] = (0.01, 0.05, 0.10, 1.0),
) -> widgets.VBox:
    if not horizons:
        raise ValueError('Select at least one measurement horizon')
    rows: list[widgets.VBox] = []
    for label, fraction in HORIZONS:
        if fraction not in horizons:
            continue
        original = render_figure(figure_name, fraction, data_dir, None)
        ablated = render_figure(figure_name, fraction, data_dir, ablated_widths)
        align_scales(original, ablated)
        pair = widgets.HBox([
            widgets.VBox([
                widgets.HTML('<b>Original: all released widths</b>'),
                plot_image(original),
            ], layout=widgets.Layout(width='630px', flex='0 0 630px')),
            widgets.VBox([
                widgets.HTML('<b>Ablated: selected widths</b>'),
                plot_image(ablated),
            ], layout=widgets.Layout(width='630px', flex='0 0 630px')),
        ], layout=widgets.Layout(width='100%', overflow_x='auto', flex_flow='row nowrap'))
        heading = 'Final saved measurement' if fraction == 1.0 else f'First {label} of saved measurements'
        rows.append(widgets.VBox([widgets.HTML(f'<h3>{heading}</h3>'), pair]))
    return widgets.VBox(rows)


# Edit these values once, then rerun the figure cells you want to refresh.
ABLATION_WIDTHS: tuple[int, ...] = SPARSE_WIDTHS
SELECTED_HORIZONS: tuple[float, ...] = (0.01, 0.05, 0.10, 1.0)


def show_figure(number: str, panels: tuple[str, ...]) -> None:
    for panel in panels:
        display(widgets.HTML(f'<h3>Figure {number}: {panel}</h3>'))
        display(make_comparison(panel, ABLATION_WIDTHS, SELECTED_HORIZONS))


## Figure 1


In [ ]:
show_figure('1', ('figure_01_resnet_15pct_final', 'figure_01_resnet_15pct_dynamics'))


## Figure 2


In [ ]:
show_figure('2', ('figure_02_resnet_15pct_test_heatmap', 'figure_02_resnet_15pct_train_heatmap'))


## Figure 4


In [ ]:
show_figure('4', ('figure_04_resnet_cifar10_noise', 'figure_04_resnet_cifar10_all_noise', 'figure_04_resnet_cifar100_noise'))


## Figure 5


In [ ]:
show_figure('5', ('figure_05_cifar10_cnn_augmentation',))


## Figure 6


In [ ]:
show_figure('6', ('figure_06_cifar10_cnn_optimizers',))


## Figure 7


In [ ]:
show_figure('7', ('figure_07_cifar100_cnn',))


## Figure 9


In [ ]:
show_figure('9', ('figure_09_resnet_20pct_panels', 'figure_09_resnet_20pct_dynamics', 'figure_09_resnet_20pct_heatmap'))


## Figure 10


In [ ]:
show_figure('10', ('figure_10_cifar10_cnn_dynamics',))


## Figure 11(a)


In [ ]:
show_figure('11(a)', ('figure_11a_cifar10_cnn_sample_sizes',))


## Figure 12


In [ ]:
show_figure('12', ('figure_12_cifar10_cnn_sample_heatmap',))


## Figure 19


In [ ]:
show_figure('19', ('figure_19_cifar100_resnet_heatmap', 'figure_19_cifar100_resnet_dynamics'))


## Figure 20


In [ ]:
show_figure('20', ('figure_20_cifar100_cnn_heatmap', 'figure_20_cifar100_cnn_dynamics'))


## Figure 21


In [ ]:
show_figure('21', ('figure_21_cifar10_cnn_weight_decay',))


## Figure 25


In [ ]:
show_figure('25', ('figure_25_cifar10_cnn_10pct_heatmap',))


## Figure 27


In [ ]:
show_figure('27', ('figure_27_cifar10_wide_cnn',))


## Released source sweeps

Each published vision source has its own cell below. These are source-level views rather than numbered paper figures. Run only the cells you want to inspect; each renders the full-width original beside the freshly generated width ablation.


### adv-mcnn-25k-p0


In [ ]:
show_figure("Source sweep", ('source_sweeps/adv-mcnn-25k-p0',))


### cifar10-mcnn-noaug-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-noaug-adam',))


### cifar10-mcnn-noaug-sgd


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-noaug-sgd',))


### cifar10-mcnn-p0-sgd-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p0-sgd-adam',))


### cifar10-mcnn-p10-sgd


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p10-sgd',))


### cifar10-mcnn-p20-sgd-rep2


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p20-sgd-rep2',))


### cifar10-mcnn-p20-sgd-rep3


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p20-sgd-rep3',))


### cifar10-mcnn-p20-sgd-rep4


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p20-sgd-rep4',))


### cifar10-mcnn-p20-sgd-rep5


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p20-sgd-rep5',))


### cifar10-mcnn-p20-sgd


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-mcnn-p20-sgd',))


### cifar10-resnet18k-50k-adam-all


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-resnet18k-50k-adam-all',))


### cifar10-resnet18k-50k-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-resnet18k-50k-adam',))


### cifar10-resnet18k-p15-adam-reps


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-resnet18k-p15-adam-reps',))


### cifar10-resnet18k-p20-decay


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar10-resnet18k-p20-decay',))


### cifar100-resnet18k-50k-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/cifar100-resnet18k-50k-adam',))


### dd_grid_p20


In [ ]:
show_figure("Source sweep", ('source_sweeps/dd_grid_p20',))


### pct-cf10-res18-50k-p05-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cf10-res18-50k-p05-adam',))


### pct-cifar10-mcnn-50000-p0-sgd-big


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-50000-p0-sgd-big',))


### pct-cifar10-mcnn-50000-p20-sgd-big


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-50000-p20-sgd-big',))


### pct-cifar10-mcnn-mdd-50k-p10-sgd-aug-fast


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-mdd-50k-p10-sgd-aug-fast',))


### pct-cifar10-mcnn-p0-adam-aug


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-p0-adam-aug',))


### pct-cifar10-mcnn-p10-sgd-aug-decay-big


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-p10-sgd-aug-decay-big',))


### pct-cifar10-mcnn-p10-sgd-aug-decay5-big


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-p10-sgd-aug-decay5-big',))


### pct-cifar10-mcnn-p20-sgd-noaug


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-mcnn-p20-sgd-noaug',))


### pct-cifar10-resnet18-50k-p15-adam-rep2


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-resnet18-50k-p15-adam-rep2',))


### pct-cifar10-resnet18-50k-p15-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-resnet18-50k-p15-adam',))


### pct-cifar10-resnet18-50k-p20-adam-decay


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar10-resnet18-50k-p20-adam-decay',))


### pct-cifar100-mcnn-p0-sgd-noaug-reps


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar100-mcnn-p0-sgd-noaug-reps',))


### pct-cifar100-mcnn-p0-sgd-noaug


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-cifar100-mcnn-p0-sgd-noaug',))


### pct-mse-resnet-p0-adam-aug


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-mse-resnet-p0-adam-aug',))


### pct-mse-resnet-p0-adam-noaug


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-mse-resnet-p0-adam-noaug',))


### pct-mse-resnet-p10-adam


In [ ]:
show_figure("Source sweep", ('source_sweeps/pct-mse-resnet-p10-adam',))


## Reading the paired plots

The left chart uses all released model widths; the right chart uses `ABLATION_WIDTHS`. Both are rendered when that figure cell runs. The ablation hides widths: it does not train new models, interpolate between widths, or average checkpoints. Early endpoint curves use the last available measurement within their cutoff; heatmaps and dynamics panels show the history through that cutoff.

Figures 1 and 7 include repeated runs. Other `Mlist` entries may encode different conditions rather than seeds. The chart layouts and source mappings are in [`short_horizon.py`](short_horizon.py); dataset caveats are in [`README.md`](README.md).
